In [16]:
import gc
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from catboost import CatBoostClassifier
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# Final evaluation: all models on train / val / test (cold/warm breakdown)

## Paths and constants

In [17]:
DATA_DIR = '/kaggle/input/datasets/f0rzah/artifacts5/'
OUTPUT_DIR = '/kaggle/working/'
RAW_DIR = '/kaggle/input/datasets/f0rzah/kion-dataset/data_original/'

CUTOFF_DATE = pd.Timestamp('2021-05-01')
WINDOW_DAYS = 14
COLD_THRESHOLD = 5
SAMPLE_SIZE = 500
SEED = 42
K = 10

## Load feature tables

In [18]:
items_features = pd.read_parquet(f'{DATA_DIR}items_features.parquet')
users_features = pd.read_parquet(f'{DATA_DIR}users_features.parquet')
print(f'items_features: {items_features.shape}')
print(f'users_features: {users_features.shape}')

items_features: (15963, 12)
users_features: (840197, 9)


## Load raw items/users (for notebook 04 scorer)

In [19]:
items_raw = pd.read_csv(f'{RAW_DIR}items.csv')[['item_id', 'content_type', 'release_year', 'age_rating', 'for_kids']].copy()
items_raw['for_kids'] = items_raw['for_kids'].fillna(0).astype('int8')

users_raw = pd.read_csv(f'{RAW_DIR}users.csv')[['user_id', 'age', 'sex', 'income', 'kids_flg']].copy()
print(f'items_raw: {items_raw.shape}')
print(f'users_raw: {users_raw.shape}')

items_raw: (15963, 5)
users_raw: (840197, 5)


## Load train/val eval samples

In [20]:
with open(f'{DATA_DIR}evaluation_samples.pkl', 'rb') as f:
    eval_data = pickle.load(f)

train_sampled_users = eval_data['train_sampled_users']
val_sampled_users = eval_data['val_sampled_users']
train_candidates = eval_data['train_candidates']
val_candidates = eval_data['val_candidates']
train_ground_truth = eval_data['train_ground_truth']
val_ground_truth = eval_data['val_ground_truth']

cold_set = set(int(u) for u in eval_data['cold_users'])
print(f'train users: {len(train_sampled_users)}, val users: {len(val_sampled_users)}')
print(f'cold_set: {len(cold_set)}')

train users: 500, val users: 500
cold_set: 452500


## Load KNN artifacts

In [21]:
with open(f'{DATA_DIR}knn_artifacts.pkl', 'rb') as f:
    knn = pickle.load(f)

print(f'knn user_history: {len(knn["user_history"])}')
print(f'knn top_500_popular: {len(knn["top_500_popular"])}')

knn user_history: 633639
knn top_500_popular: 500


## Load VAE embeddings (notebook 05)

In [22]:
user_emb_df = pd.read_parquet(f'{DATA_DIR}users_emb.parquet')
item_emb_df = pd.read_parquet(f'{DATA_DIR}items_emb.parquet')
print(f'user_emb_df: {user_emb_df.shape}')
print(f'item_emb_df: {item_emb_df.shape}')

user_emb_df: (840197, 9)
item_emb_df: (15963, 9)


## Sanity checks

In [23]:
assert len(cold_set) > 0, 'cold_set is empty'
assert len(knn['user_history']) > 0, 'knn user_history is empty'
assert len(knn['top_500_popular']) == 500, 'top_500_popular size mismatch'
assert set(train_ground_truth.keys()).issubset(set(int(u) for u in train_sampled_users))
assert set(val_ground_truth.keys()).issubset(set(int(u) for u in val_sampled_users))
assert len(user_emb_df.columns) == 1 + 8, 'VAE user emb dim != 8'
assert len(item_emb_df.columns) == 1 + 8, 'VAE item emb dim != 8'
print('sanity checks passed')

sanity checks passed


## Build test_df

In [24]:
interactions_raw = pd.read_csv(
    f'{RAW_DIR}interactions.csv',
    usecols=['user_id', 'item_id', 'last_watch_dt'],
    parse_dates=['last_watch_dt'],
)
interactions_raw = interactions_raw[interactions_raw['last_watch_dt'] >= CUTOFF_DATE].reset_index(drop=True)

max_date = interactions_raw['last_watch_dt'].max()
test_start = max_date - pd.Timedelta(days=WINDOW_DAYS)
test_df = interactions_raw[interactions_raw['last_watch_dt'] >= test_start].copy()

print(f'max_date: {max_date.date()}')
print(f'test_start: {test_start.date()}')
print(f'test_df: {test_df.shape}')

max_date: 2021-08-22
test_start: 2021-08-08
test_df: (888543, 3)


In [25]:
del interactions_raw
gc.collect()

0

## Candidate retrieval

In [26]:
def get_candidates(user_id, top_k_items=250):
    history = knn['user_history'].get(int(user_id), [])
    if len(history) < COLD_THRESHOLD:
        return knn['top_500_popular']
    history_indices = [knn['item_id_to_idx'][i] for i in history if i in knn['item_id_to_idx']]
    if not history_indices:
        return knn['top_500_popular']
    sims = cosine_similarity(knn['feature_matrix'][history_indices], knn['feature_matrix'])
    top_idx = np.argsort(sims.mean(axis=0))[::-1][:top_k_items]
    similar_items = [knn['item_ids'][i] for i in top_idx]
    candidates = list(dict.fromkeys(similar_items + knn['top_500_popular']))
    history_set = set(history)
    candidates = [i for i in candidates if i not in history_set]
    return candidates[:500]

## Build test evaluation samples

In [27]:
test_users_all = test_df['user_id'].unique()
n_train = np.array([len(knn['user_history'].get(int(u), [])) for u in test_users_all])
is_cold = n_train < COLD_THRESHOLD

cold_pool = test_users_all[is_cold]
warm_pool = test_users_all[~is_cold]
test_cold_set = set(int(u) for u in cold_pool)
print(f'cold_pool: {len(cold_pool)}, warm_pool: {len(warm_pool)}')

cold_pool: 215889, warm_pool: 63240


In [28]:
ratio = len(cold_pool) / len(test_users_all)
n_cold = min(int(SAMPLE_SIZE * ratio), len(cold_pool))
n_warm = min(SAMPLE_SIZE - n_cold, len(warm_pool))

np.random.seed(SEED)
test_sampled_users = np.concatenate([
    np.random.choice(cold_pool, n_cold, replace=False),
    np.random.choice(warm_pool, n_warm, replace=False),
])
print(f'test_sampled_users: {len(test_sampled_users)} (cold {n_cold}, warm {n_warm})')

test_sampled_users: 500 (cold 386, warm 114)


In [29]:
test_ground_truth = test_df[test_df['user_id'].isin(test_sampled_users)].groupby('user_id')['item_id'].apply(list).to_dict()
test_candidates = {int(u): get_candidates(u) for u in tqdm(test_sampled_users, desc='test candidates')}
print(f'test_ground_truth: {len(test_ground_truth)}')

test candidates: 100%|██████████| 500/500 [00:00<00:00, 638.06it/s] 

test_ground_truth: 500


In [30]:
with open(f'{OUTPUT_DIR}test_eval_samples.pkl', 'wb') as f:
    pickle.dump({
        'test_sampled_users': test_sampled_users,
        'test_candidates': test_candidates,
        'test_ground_truth': test_ground_truth,
        'test_cold_set': test_cold_set,
        'test_start': test_start,
        'max_date': max_date,
        'seed': SEED,
    }, f)
print('test_eval_samples.pkl saved')

test_eval_samples.pkl saved


In [31]:
del test_df
gc.collect()

0

## Load Two-Tower model

In [32]:
class UserTower(nn.Module):
    def __init__(self, vocab_sizes, emb_dim=16, hidden_dim=128, out_dim=64):
        super().__init__()
        self.emb_age = nn.Embedding(vocab_sizes['age'], emb_dim)
        self.emb_sex = nn.Embedding(vocab_sizes['sex'], emb_dim // 2)
        self.emb_income = nn.Embedding(vocab_sizes['income'], emb_dim)
        in_dim = emb_dim + emb_dim // 2 + emb_dim + 2
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, out_dim), nn.ReLU(),
        )
    def forward(self, u_age, u_sex, u_income, u_kids, u_interactions):
        x = torch.cat([
            self.emb_age(u_age), self.emb_sex(u_sex), self.emb_income(u_income),
            u_kids.unsqueeze(1), u_interactions.unsqueeze(1),
        ], dim=1)
        return self.mlp(x)

In [33]:
class ItemTower(nn.Module):
    def __init__(self, vocab_sizes, emb_dim=16, hidden_dim=128, out_dim=64):
        super().__init__()
        self.emb_type = nn.Embedding(vocab_sizes['content_type'], emb_dim)
        self.emb_decade = nn.Embedding(vocab_sizes['release_decade'], emb_dim)
        in_dim = emb_dim * 2 + 8
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim, out_dim), nn.ReLU(),
        )
    def forward(self, i_type, i_decade, i_age_rating, i_genre_score, i_country_score,
                i_actor_score, i_director_score, i_pop_log, i_avg_dur_log, i_kids_cross):
        x = torch.cat([
            self.emb_type(i_type), self.emb_decade(i_decade),
            i_age_rating.unsqueeze(1), i_genre_score.unsqueeze(1), i_country_score.unsqueeze(1),
            i_actor_score.unsqueeze(1), i_director_score.unsqueeze(1),
            i_pop_log.unsqueeze(1), i_avg_dur_log.unsqueeze(1), i_kids_cross.unsqueeze(1),
        ], dim=1)
        return self.mlp(x)

In [34]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim=64, hidden_dim=32, out_dim=32):
        super().__init__()
        self.projector = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
        return self.projector(x)

In [35]:
class TwoTowerModel(nn.Module):
    def __init__(self, vocab_sizes, emb_dim=16, hidden_dim=128, out_dim=64, proj_dim=32):
        super().__init__()
        self.user_tower = UserTower(vocab_sizes, emb_dim, hidden_dim, out_dim)
        self.item_tower = ItemTower(vocab_sizes, emb_dim, hidden_dim, out_dim)
        self.user_projection = ProjectionHead(out_dim, out_dim // 2, proj_dim)
        self.item_projection = ProjectionHead(out_dim, out_dim // 2, proj_dim)
    def forward(self, batch, return_proj=True):
        u_emb = self.user_tower(
            batch['u_age'], batch['u_sex'], batch['u_income'],
            batch['u_kids'], batch['u_interactions'],
        )
        i_emb = self.item_tower(
            batch['i_type'], batch['i_decade'], batch['i_age_rating'],
            batch['i_genre_score'], batch['i_country_score'], batch['i_actor_score'],
            batch['i_director_score'], batch['i_pop_log'], batch['i_avg_dur_log'],
            batch['i_kids_cross'],
        )
        if return_proj:
            return u_emb, i_emb, self.user_projection(u_emb), self.item_projection(i_emb)
        return u_emb, i_emb

In [36]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt = torch.load(f'{DATA_DIR}two_tower_model.pt', map_location='cpu')
tt_vocabularies = ckpt['vocabularies']
tt_vocab_sizes = ckpt['vocab_sizes']

tt_model = TwoTowerModel(vocab_sizes=tt_vocab_sizes, **ckpt['config'])
tt_model.load_state_dict(ckpt['state_dict'])
tt_model = tt_model.to(DEVICE)
tt_model.eval()
print('two_tower loaded')

two_tower loaded


## Load CatBoost models

In [37]:
cb_baseline = CatBoostClassifier().load_model(f'{DATA_DIR}catboost_model.cbm')
cb_raw = CatBoostClassifier().load_model(f'{DATA_DIR}catboost_raw_features.cbm')
cb_vae = CatBoostClassifier().load_model(f'{DATA_DIR}catboost_vae.cbm')

FEATURES_02 = cb_baseline.feature_names_
FEATURES_RAW = cb_raw.feature_names_
FEATURES_VAE = cb_vae.feature_names_

print(f'02: {len(FEATURES_02)} features')
print(f'04: {len(FEATURES_RAW)} features')
print(f'05: {len(FEATURES_VAE)} features')

02: 20 features
04: 8 features
05: 28 features


In [38]:
CAT_02 = [c for c in FEATURES_02 if c in ['content_type', 'release_decade', 'age', 'sex', 'income']]
CAT_04 = ['content_type', 'age', 'sex', 'income']
CAT_VAE = [c for c in FEATURES_VAE if c in ['content_type', 'release_decade', 'age', 'sex', 'income']]

## Extract Two-Tower embeddings

In [39]:
def extract_tt_user_embs(model, users_features, vocab, device, batch_size=1024):
    model.eval()
    out, ids = [], []
    for i in range(0, len(users_features), batch_size):
        df = users_features.iloc[i:i + batch_size]
        n = len(df)
        batch = {
            'u_age': torch.tensor(df['age'].fillna('Unknown').astype(str).map(vocab['age']).fillna(0).values, dtype=torch.long, device=device),
            'u_sex': torch.tensor(df['sex'].fillna('Unknown').astype(str).map(vocab['sex']).fillna(0).values, dtype=torch.long, device=device),
            'u_income': torch.tensor(df['income'].fillna('Unknown').astype(str).map(vocab['income']).fillna(0).values, dtype=torch.long, device=device),
            'u_kids': torch.tensor(df['kids_flg'].fillna(0).values, dtype=torch.float32, device=device),
            'u_interactions': torch.tensor(df['user_total_interactions_log'].fillna(0).values, dtype=torch.float32, device=device),
            'i_type': torch.zeros(n, dtype=torch.long, device=device),
            'i_decade': torch.zeros(n, dtype=torch.long, device=device),
            'i_age_rating': torch.zeros(n, device=device),
            'i_genre_score': torch.zeros(n, device=device),
            'i_country_score': torch.zeros(n, device=device),
            'i_actor_score': torch.zeros(n, device=device),
            'i_director_score': torch.zeros(n, device=device),
            'i_pop_log': torch.zeros(n, device=device),
            'i_avg_dur_log': torch.zeros(n, device=device),
            'i_kids_cross': torch.zeros(n, device=device),
        }
        with torch.no_grad():
            u_emb, _ = model(batch, return_proj=False)
        out.append(F.normalize(u_emb, p=2, dim=1).cpu().numpy())
        ids.extend(df['user_id'].values)
    return np.vstack(out), np.array(ids)

In [40]:
def extract_tt_item_embs(model, items_features, vocab, device, batch_size=1024):
    model.eval()
    out, ids = [], []
    for i in range(0, len(items_features), batch_size):
        df = items_features.iloc[i:i + batch_size]
        n = len(df)
        batch = {
            'i_type': torch.tensor(df['content_type'].fillna('Unknown').astype(str).map(vocab['content_type']).fillna(0).values, dtype=torch.long, device=device),
            'i_decade': torch.tensor(df['release_decade'].fillna('Unknown').astype(str).map(vocab['release_decade']).fillna(0).values, dtype=torch.long, device=device),
            'i_age_rating': torch.tensor(df['age_rating_ord'].fillna(2).values, dtype=torch.float32, device=device),
            'i_genre_score': torch.tensor(df['genre_score'].fillna(0).values, dtype=torch.float32, device=device),
            'i_country_score': torch.tensor(df['country_score'].fillna(0).values, dtype=torch.float32, device=device),
            'i_actor_score': torch.tensor(df['actor_score'].fillna(0).values, dtype=torch.float32, device=device),
            'i_director_score': torch.tensor(df['director_score'].fillna(0).values, dtype=torch.float32, device=device),
            'i_pop_log': torch.tensor(df['item_popularity_log'].fillna(0).values, dtype=torch.float32, device=device),
            'i_avg_dur_log': torch.tensor(df['item_avg_duration_log'].fillna(0).values, dtype=torch.float32, device=device),
            'i_kids_cross': torch.zeros(n, device=device),
            'u_age': torch.zeros(n, dtype=torch.long, device=device),
            'u_sex': torch.zeros(n, dtype=torch.long, device=device),
            'u_income': torch.zeros(n, dtype=torch.long, device=device),
            'u_kids': torch.zeros(n, device=device),
            'u_interactions': torch.zeros(n, device=device),
        }
        with torch.no_grad():
            _, i_emb = model(batch, return_proj=False)
        out.append(F.normalize(i_emb, p=2, dim=1).cpu().numpy())
        ids.extend(df['item_id'].values)
    return np.vstack(out), np.array(ids)

In [41]:
user_tt_embs, user_tt_ids = extract_tt_user_embs(tt_model, users_features, tt_vocabularies, DEVICE)
item_tt_embs, item_tt_ids = extract_tt_item_embs(tt_model, items_features, tt_vocabularies, DEVICE)

user_tt_dict = {int(u): e for u, e in zip(user_tt_ids, user_tt_embs)}
item_tt_dict = {int(i): e for i, e in zip(item_tt_ids, item_tt_embs)}
print(f'user_tt: {user_tt_embs.shape}, item_tt: {item_tt_embs.shape}')

user_tt: (840197, 64), item_tt: (15963, 64)


## Scorers

In [42]:
def score_02(df):
    out = df.merge(users_features, on='user_id', how='left')
    out = out.merge(items_features, on='item_id', how='left')
    out['is_kids_user_and_kids_content'] = ((out['kids_flg'] == 1) & (out['age_rating_ord'] <= 1)).astype('int8')
    for c in CAT_02:
        out[c] = out[c].astype(str)
    for c in FEATURES_02:
        if c not in CAT_02:
            out[c] = out[c].fillna(0.0).astype('float32')
    out['score'] = cb_baseline.predict_proba(out[FEATURES_02])[:, 1]
    return out[['user_id', 'item_id', 'score']]

In [43]:
def score_04(df):
    out = df.merge(users_raw, on='user_id', how='left')
    out = out.merge(items_raw, on='item_id', how='left')
    for c in CAT_04:
        out[c] = out[c].astype(str)
    out['for_kids'] = out['for_kids'].fillna(0).astype('int8')
    out['score'] = cb_raw.predict_proba(out[FEATURES_RAW])[:, 1]
    return out[['user_id', 'item_id', 'score']]

In [44]:
TT_ZERO = np.zeros(user_tt_embs.shape[1], dtype=np.float32)

def score_03(df):
    u = np.stack([user_tt_dict.get(int(x), TT_ZERO) for x in df['user_id'].values])
    i = np.stack([item_tt_dict.get(int(x), TT_ZERO) for x in df['item_id'].values])
    out = df[['user_id', 'item_id']].copy()
    out['score'] = (u * i).sum(axis=1)
    return out

In [45]:
def score_05(df):
    out = df.merge(users_features, on='user_id', how='left')
    out = out.merge(items_features, on='item_id', how='left')
    out = out.merge(user_emb_df, on='user_id', how='left')
    out = out.merge(item_emb_df, on='item_id', how='left')
    out['is_kids_user_and_kids_content'] = ((out['kids_flg'] == 1) & (out['age_rating_ord'] <= 1)).astype('int8')
    for c in CAT_VAE:
        out[c] = out[c].fillna('Unknown').astype(str)
    for c in FEATURES_VAE:
        if c not in CAT_VAE:
            out[c] = out[c].fillna(0.0).astype('float32')
    out['score'] = cb_vae.predict_proba(out[FEATURES_VAE])[:, 1]
    return out[['user_id', 'item_id', 'score']]

## Metrics (per-user)

In [46]:
def compute_metrics(top_k_items, true_items, k=K):
    true_set = set(true_items)
    hits = 0
    sum_precision = 0.0
    sum_ndcg = 0.0
    for rank, item in enumerate(top_k_items[:k], start=1):
        if item in true_set:
            hits += 1
            sum_precision += hits / rank
            sum_ndcg += 1.0 / np.log2(rank + 1)
    num_relevant = min(k, len(true_set))
    if num_relevant == 0:
        return 0.0, 0.0, 0.0
    map_ = sum_precision / num_relevant
    ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, num_relevant + 1))
    ndcg = sum_ndcg / ideal_dcg if ideal_dcg > 0 else 0.0
    hr = 1.0 if hits > 0 else 0.0
    return hr, ndcg, map_

In [47]:
def evaluate_per_user(scorer, sampled_users, candidates, ground_truth, batch_size=20, k=K):
    users_arr, hr_arr, ndcg_arr, map_arr = [], [], [], []
    for i in tqdm(range(0, len(sampled_users), batch_size), leave=False):
        batch = sampled_users[i:i + batch_size]
        rows = [{'user_id': int(u), 'item_id': iid} for u in batch for iid in candidates[int(u)]]
        df = pd.DataFrame(rows)
        df = scorer(df)
        for u in batch:
            u_int = int(u)
            preds = df[df['user_id'] == u_int].sort_values('score', ascending=False)
            hr, ndcg, map_ = compute_metrics(preds['item_id'].head(k).tolist(), ground_truth.get(u_int, []), k=k)
            users_arr.append(u_int)
            hr_arr.append(hr)
            ndcg_arr.append(ndcg)
            map_arr.append(map_)
    return np.array(users_arr), np.array(hr_arr), np.array(ndcg_arr), np.array(map_arr)

In [48]:
def summarize(users_arr, hr, ndcg, map_, cold_set):
    is_cold = np.isin(users_arr, list(cold_set))
    rows = []
    for group, mask in [('all', np.ones_like(is_cold, dtype=bool)), ('cold', is_cold), ('warm', ~is_cold)]:
        n = int(mask.sum())
        if n == 0:
            rows.append({'Group': group, 'N': 0, 'HR@10': np.nan, 'NDCG@10': np.nan, 'MAP@10': np.nan, 'MAP_median': np.nan, 'MAP_p25': np.nan, 'MAP_p75': np.nan})
            continue
        rows.append({
            'Group': group,
            'N': n,
            'HR@10': float(hr[mask].mean()),
            'NDCG@10': float(ndcg[mask].mean()),
            'MAP@10': float(map_[mask].mean()),
            'MAP_median': float(np.median(map_[mask])),
            'MAP_p25': float(np.percentile(map_[mask], 25)),
            'MAP_p75': float(np.percentile(map_[mask], 75)),
        })
    return pd.DataFrame(rows)

In [49]:
def run_eval(scorer, model_name, split_name, sampled_users, candidates, ground_truth, cold_set):
    users_arr, hr, ndcg, map_ = evaluate_per_user(scorer, sampled_users, candidates, ground_truth)
    df = summarize(users_arr, hr, ndcg, map_, cold_set)
    df.insert(0, 'Split', split_name)
    df.insert(0, 'Model', model_name)
    return df

## Evaluate 02 — CatBoost baseline

In [50]:
r02_train = run_eval(score_02, '02_catboost_baseline', 'train', train_sampled_users, train_candidates, train_ground_truth, cold_set)
r02_val = run_eval(score_02, '02_catboost_baseline', 'val', val_sampled_users, val_candidates, val_ground_truth, cold_set)
r02_test = run_eval(score_02, '02_catboost_baseline', 'test', test_sampled_users, test_candidates, test_ground_truth, test_cold_set)

## Evaluate 04 — CatBoost raw features

In [51]:
r04_train = run_eval(score_04, '04_catboost_raw', 'train', train_sampled_users, train_candidates, train_ground_truth, cold_set)
r04_val = run_eval(score_04, '04_catboost_raw', 'val', val_sampled_users, val_candidates, val_ground_truth, cold_set)
r04_test = run_eval(score_04, '04_catboost_raw', 'test', test_sampled_users, test_candidates, test_ground_truth, test_cold_set)

## Evaluate 03 — Two-Tower

In [52]:
r03_train = run_eval(score_03, '03_two_tower', 'train', train_sampled_users, train_candidates, train_ground_truth, cold_set)
r03_val = run_eval(score_03, '03_two_tower', 'val', val_sampled_users, val_candidates, val_ground_truth, cold_set)
r03_test = run_eval(score_03, '03_two_tower', 'test', test_sampled_users, test_candidates, test_ground_truth, test_cold_set)

## Evaluate 05 — VAE + CatBoost

In [53]:
r05_train = run_eval(score_05, '05_vae_catboost', 'train', train_sampled_users, train_candidates, train_ground_truth, cold_set)
r05_val = run_eval(score_05, '05_vae_catboost', 'val', val_sampled_users, val_candidates, val_ground_truth, cold_set)
r05_test = run_eval(score_05, '05_vae_catboost', 'test', test_sampled_users, test_candidates, test_ground_truth, test_cold_set)

## Summary — train

In [54]:
summary_train = pd.concat([r02_train, r03_train, r04_train, r05_train], ignore_index=True)
print(summary_train.to_string(index=False))

               Model Split Group   N    HR@10  NDCG@10   MAP@10  MAP_median  MAP_p25  MAP_p75
02_catboost_baseline train   all 500 0.426000 0.232473 0.176954    0.000000      0.0 0.250000
02_catboost_baseline train  cold 357 0.596639 0.325593 0.247834    0.125000      0.0 0.500000
02_catboost_baseline train  warm 143 0.000000 0.000000 0.000000    0.000000      0.0 0.000000
        03_two_tower train   all 500 0.244000 0.115863 0.078783    0.000000      0.0 0.000000
        03_two_tower train  cold 357 0.341737 0.162272 0.110340    0.000000      0.0 0.142857
        03_two_tower train  warm 143 0.000000 0.000000 0.000000    0.000000      0.0 0.000000
     04_catboost_raw train   all 500 0.294000 0.177701 0.146750    0.000000      0.0 0.166667
     04_catboost_raw train  cold 357 0.411765 0.248881 0.205532    0.000000      0.0 0.333333
     04_catboost_raw train  warm 143 0.000000 0.000000 0.000000    0.000000      0.0 0.000000
     05_vae_catboost train   all 500 0.418000 0.199473 0.139

## Summary — val

In [55]:
summary_val = pd.concat([r02_val, r03_val, r04_val, r05_val], ignore_index=True)
print(summary_val.to_string(index=False))

               Model Split Group   N    HR@10  NDCG@10   MAP@10  MAP_median  MAP_p25  MAP_p75
02_catboost_baseline   val   all 500 0.456000 0.217699 0.161682         0.0      0.0 0.200000
02_catboost_baseline   val  cold 120 0.375000 0.175725 0.128379         0.0      0.0 0.166667
02_catboost_baseline   val  warm 380 0.481579 0.230954 0.172199         0.0      0.0 0.213056
        03_two_tower   val   all 500 0.214000 0.086355 0.056462         0.0      0.0 0.000000
        03_two_tower   val  cold 120 0.183333 0.066537 0.039929         0.0      0.0 0.000000
        03_two_tower   val  warm 380 0.223684 0.092614 0.061683         0.0      0.0 0.000000
     04_catboost_raw   val   all 500 0.330000 0.163693 0.126639         0.0      0.0 0.114583
     04_catboost_raw   val  cold 120 0.266667 0.132370 0.101364         0.0      0.0 0.057292
     04_catboost_raw   val  warm 380 0.350000 0.173584 0.134620         0.0      0.0 0.125000
     05_vae_catboost   val   all 500 0.452000 0.176230 0.112

## Summary — test

In [56]:
summary_test = pd.concat([r02_test, r03_test, r04_test, r05_test], ignore_index=True)
print(summary_test.to_string(index=False))

               Model Split Group   N    HR@10  NDCG@10   MAP@10  MAP_median  MAP_p25  MAP_p75
02_catboost_baseline  test   all 500 0.382000 0.164083 0.114642         0.0      0.0 0.125000
02_catboost_baseline  test  cold 386 0.409326 0.184478 0.131195         0.0      0.0 0.166667
02_catboost_baseline  test  warm 114 0.289474 0.095026 0.058595         0.0      0.0 0.034722
        03_two_tower  test   all 500 0.194000 0.067267 0.041123         0.0      0.0 0.000000
        03_two_tower  test  cold 386 0.238342 0.085332 0.052747         0.0      0.0 0.000000
        03_two_tower  test  warm 114 0.043860 0.006099 0.001764         0.0      0.0 0.000000
     04_catboost_raw  test   all 500 0.242000 0.109486 0.081491         0.0      0.0 0.000000
     04_catboost_raw  test  cold 386 0.259067 0.129042 0.099790         0.0      0.0 0.033333
     04_catboost_raw  test  warm 114 0.184211 0.043271 0.019531         0.0      0.0 0.000000
     05_vae_catboost  test   all 500 0.374000 0.135019 0.083

## Save summary

In [57]:
summary = pd.concat([summary_train, summary_val, summary_test], ignore_index=True)
summary.to_csv(f'{OUTPUT_DIR}final_evaluation_summary.csv', index=False)
print(f'summary rows: {len(summary)}')
print('saved to final_evaluation_summary.csv')

summary rows: 36
saved to final_evaluation_summary.csv
